In [ ]:
import networkx as nx
import os
import re

# 📌 Met le bon chemin vers un fichier CFG
cfg_path = "cfg1.json"  # Nom trompeur mais c'est un DOT !

# Regex pré-compilées
EDGE_REGEX = re.compile(r'"([a-zA-Z0-9]+)"\s*->\s*"([a-zA-Z0-9]+)"')
NODE_REGEX = re.compile(r'"([a-zA-Z0-9]+)"\s*\[label\s*=\s*"(.*?)"\s*\]')


def parse_cfg_file(cfg_path):
    try:
        with open(cfg_path, "r", encoding="utf-8", errors="ignore") as f:
            return [line.strip() for line in f if line.strip()]
    except Exception:
        return []
    
directory = "/Users/nouroudine/Documents/Challenge_Sorbonne_2025/Assembly_Model/cfg_files"

def extract_features(name, directory):
    cfg_path = os.path.join(directory, f"{name}.json")
    
    # Initialisation des variables de stockage
    assembly = []
    assembly_inst = []
    opcode = []
    adress = []
    undef = []
    dead0f = []

    # Vérification de l'existence du fichier
    if os.path.exists(cfg_path):
        try:
            G = nx.DiGraph()

            # Parse the file
            file = parse_cfg_file(cfg_path)

            for line in file:
                # Traitement des arêtes
                if '->' in line:
                    edge_match = EDGE_REGEX.match(line)
                    if edge_match:
                        src, dst = edge_match.groups()
                        G.add_edge(src, dst)

                # Traitement des nœuds
                if "[" in line and "]" in line:
                    node_match = NODE_REGEX.match(line)
                    if node_match:
                        node, label = node_match.groups()
                        G.add_node(node, label=label)

            # Extraire les informations des nœuds du graph
            for node in G.nodes:
                label = G.nodes[node].get('label', "").split(':')  # Séparer le label par ':'

                if len(label) <= 2:
                    undef.append(label[-1].strip().strip('"'))
                    dead0f.append(label[0].strip().strip('"'))
                    continue

                if len(label) > 3:
                    # Extraction des informations détaillées pour chaque instruction
                    assembly.append(label[-2].strip().strip('"') + label[-1].strip('"'))
                    assembly_inst.append(label[-1].strip().strip('"').split(" ")[0])
                    opcode.append(label[-3].strip().strip('"'))
                    adress.append(label[0].strip().strip('"'))

                if len(label) < 4:
                    # Cas où le label contient moins de 4 éléments, récupération des éléments disponibles
                    assembly.append(label[-1].strip().strip('"'))  # L'instruction complète est le dernier élément
                    assembly_inst.append(label[-1].strip().strip('"').split(" ")[0])
                    opcode.append(label[-2].strip().strip('"'))
                    adress.append(label[0].strip().strip('"'))
        except Exception as e:
            print(f"Error processing {cfg_path}: {e}")
            
        # Si tout a bien été exécuté, on renvoie les valeurs extraites
        return G, assembly, assembly_inst, opcode, adress, undef, dead0f
    else:
        print(f"File {cfg_path} does not exist.")
        return None


In [4]:
file = "0ae65ce51eaff6305ee085b85a39e25ffb6fe8e707dcd68775aaa8ba7ee9c136"
G, assembly, assembly_inst, opcode, adress, undef, dead0f = extract_features(file, directory)
print(f"Assembly: {assembly[:5]}")
print(f"Assembly Instructions: {assembly_inst[:5]}")
print(f"Opcode: {opcode[:5]}")
print(f"Adress: {adress[:5]}")
print(f"Undef: {undef[:5]}")
print(f"Dead0f: {dead0f[:5]}")

Assembly: ['mov qword ptr [rsp + 8], rcx', 'add rsp, 0x28', 'mov qword ptr [rsp + 8], rcx', 'mov qword ptr [rsp + 0x10], rdx', 'movzx eax, al']
Assembly Instructions: ['mov', 'add', 'mov', 'mov', 'movzx']
Opcode: ['CALL', 'RET', 'CALL', 'CALL', 'JCC']
Adress: ['140001000', '140001013', '140005714', '140001018', '140001030']
Undef: ['UNDEF']
Dead0f: ['dead0def']


In [178]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
vectorizer = TfidfVectorizer(analyzer="word", ngram_range=(1,2), max_features=50)
X = vectorizer.fit_transform(assembly)  # assembly = liste de toutes les instructions
df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
print(df.head())
df.to_csv("assembly.csv", index=False)

   0x1  0x20  0x30  0x64  add   al  byte  byte ptr  call  cmp  ...       rbx  \
0  0.0   0.0   0.0   0.0  0.0  0.0   0.0       0.0   0.0  0.0  ...  1.000000   
1  0.0   0.0   0.0   0.0  0.0  0.0   0.0       0.0   0.0  0.0  ...  0.480921   
2  0.0   0.0   0.0   0.0  0.0  0.0   0.0       0.0   0.0  0.0  ...  0.000000   
3  0.0   0.0   0.0   0.0  0.0  0.0   0.0       0.0   0.0  0.0  ...  1.000000   
4  0.0   0.0   0.0   0.0  0.0  0.0   0.0       0.0   0.0  0.0  ...  0.480921   

   rcx  rcx qword  rdi  rdx       rip  rsi  rsp  test  xor  
0  0.0        0.0  0.0  0.0  0.000000  0.0  0.0   0.0  0.0  
1  0.0        0.0  0.0  0.0  0.000000  0.0  0.0   0.0  0.0  
2  0.0        0.0  0.0  0.0  0.687753  0.0  0.0   0.0  0.0  
3  0.0        0.0  0.0  0.0  0.000000  0.0  0.0   0.0  0.0  
4  0.0        0.0  0.0  0.0  0.000000  0.0  0.0   0.0  0.0  

[5 rows x 50 columns]


In [77]:
import pandas as pd
df0 = pd.read_csv("/Users/nouroudine/Documents/Challenge_Sorbonne_2025/training_set_metadata.csv", sep=";")
labels = list(df0.iloc[:1, 1:])
print(labels)

['64-bit execution via heavens gate', '64bits', 'PEB access', 'accept command line arguments', 'access the Windows event log', 'act as TCP client', 'allocate RW memory', 'allocate RWX memory', 'allocate memory', 'allocate or change RW memory', 'allocate or change RWX memory', 'allocate thread local storage', 'android', 'anorganix', 'apatch', 'apk', 'arm', 'armadillo', 'aspack', 'asprotect', 'assembly', 'attach user process memory', 'attachment', 'authenticate HMAC', 'block operations on executable memory pages using Arbitrary Code Guard', 'bobsoft', 'bypass Windows File Protection', 'calculate modulo 256 via x86 assembly', 'calls-wmi', 'capture screenshot', 'capture screenshot in Go', 'cexe', 'change file permission on Linux', 'change memory protection', 'change the wallpaper', 'check HTTP status code', 'check OS version', 'check ProcessDebugFlags', 'check ProcessDebugPort', 'check SystemKernelDebuggerInformation', 'check for OutputDebugString error', 'check for PEB BeingDebugged flag'

In [ ]:
'''
import shutil

# Liste des 100 premiers noms de fichiers dans df['names']
names = df['name'][200:300]

# Chemins des répertoires source et destination
source_dir = "/Users/nouroudine/Documents/Challenge Sorbonne 2025/folder_training_set"
dest_dir = "/Users/nouroudine/Documents/Challenge Sorbonne 2025/Assembly_Model/cfg_files"
i = 0
for name in names:
    source_path = os.path.join(source_dir, f"{name}.json")  # Chemin du fichier source
    dest_path = os.path.join(dest_dir, f"{name}.json")  # Chemin de destination
    
    shutil.copy(source_path, dest_path)
    
df = df[df['name'].isin(names)]
df.reset_index(drop=True, inplace=True)
df
'''

In [180]:
df = pd.read_csv("/Users/nouroudine/Documents/Challenge_Sorbonne_2025/Assembly_Model/df_assembly_100.csv")

In [181]:
df.head()

,name,64-bit execution via heavens gate,64bits,PEB access,accept command line arguments,access the Windows event log,act as TCP client,allocate RW memory,allocate RWX memory,allocate memory,...,winzip,wise,worm,write and execute a file,write clipboard data,write file on Linux,write file on Windows,write pipe,xorcrypt,yoda
0,c8e592c3dbec667f85d6e311357590cd67407a403eb7c1...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,b69c8f405b8435b11becdb38782bdcd03b021b132182fd...,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,f81896373a5929b3bf9734ba404198138043ec70312e54...,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,a82c9123c12957ef853f22cbdf6656194956620d486a4b...,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,df57df85d9ecc43daea5974e440919451a58b744f9db40...,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
from tqdm import tqdm

# Initialiser des listes vides pour stocker les résultats
graphs, assemblies, assembly_insts, opcodes, adresses, undefs, dead0fs = [], [], [], [], [], [], []

# Parcourir chaque nom de fichier dans df['name'] avec une barre de progression
for name in tqdm(df['name'], desc="Processing files", unit="file"):
    # Extraire les caractéristiques pour chaque fichier
    G, assembly, assembly_inst, opcode, adress, undef, dead0f = extract_features(name, directory)
    
    # Ajouter les résultats extraits aux listes correspondantes
    graphs.append(G)
    assemblies.append(assembly)
    assembly_insts.append(assembly_inst)
    opcodes.append(opcode)
    adresses.append(adress)
    undefs.append(undef)
    dead0fs.append(dead0f)


Processing files: 100%|██████████| 100/100 [00:50<00:00,  1.97file/s]


In [182]:
print(f"df['name']: {len(df['name'])}, assemblies: {len(assemblies)}")


df['name']: 100, assemblies: 100


In [183]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
for i, name in enumerate(df['name'].values):
    vectorizer = TfidfVectorizer(analyzer="word", ngram_range=(1,2), max_features=1000)
    X = vectorizer.fit_transform(assemblies[i])  # assembly = liste de toutes les instructions
    df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
    df.to_csv(f"{name}.csv", index=False)

In [1]:
list_df = []
for name in os.listdir("/Users/nouroudine/Documents/Challenge_Sorbonne_2025/Assembly_Model/assembly_csv"):
    try:
        # Essaye de lire avec l'encodage utf-8, et si ça échoue, essaye 'latin1' ou 'ISO-8859-1'
        df = pd.read_csv(f"/Users/nouroudine/Documents/Challenge_Sorbonne_2025/Assembly_Model/assembly_csv/{name}", encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(f"/Users/nouroudine/Documents/Challenge_Sorbonne_2025/Assembly_Model/assembly_csv/{name}", encoding='ISO-8859-1')
    list_df.append(df)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [102]:
for name, i in zip(os.listdir("/Users/nouroudine/Documents/Challenge_Sorbonne_2025/Assembly_Model/cfg_files"), range(100)):
	list_df[i]['name'] = name

In [104]:
df_final = pd.concat(list_df, axis=0).fillna(0)

In [115]:
df_final = pd.read_csv("/Users/nouroudine/Documents/Challenge_Sorbonne_2025/Assembly_Model/df_assembly_final.csv")

In [116]:
df_final['name'] = df_final['name'].str.split('.').str[0]
df_combined = pd.merge(df, df_final, on='name', how='left')

In [ ]:
df_combined.to_csv("/Users/nouroudine/Documents/Challenge_Sorbonne_2025/Assembly_Model/df_assembly_combined.csv", index=False)

In [120]:
df_final.head()

,0x0,0x0 call,0x0 jmp,0x0 mov,0x0 push,0x0 topstack,0x1,0x1 call,0x1 jmp,0x10,...,r8b 0x4,r8d 0x64,r8d 0xc000,r9b 0x4,r9d 0x7fff,r9d cmp,r9d r12,rax 0x140,rbx 0x1a0,rdx 0xa
0,0.017454,0.006400,0.000582,0.001745,0.005236,0.003491,0.002909,0.001164,0.001164,0.013963,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.011467,0.000000,0.000000,0.002854,0.001841,0.000000,0.040276,0.000760,0.000000,0.046795,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.006537,0.000000,0.000000,0.000769,0.002692,0.000000,0.003845,0.000000,0.000000,0.015765,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.007715,0.000441,0.000000,0.002204,0.000661,0.000000,0.016753,0.000441,0.000441,0.037914,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.046469,0.001936,0.002420,0.015974,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [136]:
features = [e for e in list(df_final.columns) if e not in labels and e != 'name']


In [172]:
from sklearn.model_selection import train_test_split

# Séparer les features et labels
X = df_combined[features]
X = X.loc[:, X.nunique() >= 50]
y = df_combined[labels]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Vérification des dimensions
print(f"Taille train : {X_train.shape}, Taille test : {X_test.shape}")


Taille train : (80, 213), Taille test : (20, 213)


In [173]:
X

,0x0,0x1,0x10,0x10 mov,0x14,0x18,0x18 mov,0x1c,0x1f,0x2,...,lea ecx,mov byte,nop,or eax,ptr ebx,ptr edx,qword,qword ptr,sbb,si
0,0.003656,0.012243,0.017182,0.007751,0.025727,0.010937,0.003775,0.009089,0.002004,0.007824,...,0.008459,0.006482,0.001493,0.001470,0.020492,0.023810,0.003519,0.003519,0.003246,0.009262
1,0.013310,0.014780,0.041055,0.008557,0.019101,0.017113,0.003112,0.013570,0.005099,0.020657,...,0.013483,0.009248,0.000000,0.008643,0.018583,0.010890,0.001815,0.001815,0.000000,0.002506
2,0.000000,0.099346,0.033115,0.004139,0.000000,0.000000,0.000000,0.004139,0.000000,0.008279,...,0.000000,0.000000,0.000000,0.000000,0.033115,0.000000,0.000000,0.000000,0.000000,0.004139
3,0.041302,0.061446,0.039271,0.007899,0.021723,0.015912,0.004175,0.020933,0.002990,0.007166,...,0.016927,0.002257,0.000000,0.001975,0.011398,0.004965,0.004852,0.004852,0.001072,0.000903
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.015917,0.027054,0.040890,0.010427,0.041255,0.032971,0.010843,0.027106,0.001751,0.007364,...,0.011343,0.008972,0.029861,0.001450,0.013392,0.007420,0.001930,0.001930,0.001930,0.000558
96,0.005027,0.009822,0.011785,0.001991,0.007498,0.005108,0.000878,0.003556,0.001739,0.005519,...,0.006608,0.037232,0.001414,0.001959,0.005787,0.007189,0.066779,0.066779,0.001398,0.000524
97,0.000000,0.006748,0.013497,0.006748,0.006748,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.006748,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
98,0.015105,0.047725,0.036806,0.009296,0.026691,0.021838,0.005451,0.016746,0.001777,0.011107,...,0.006989,0.017754,0.000649,0.002563,0.020197,0.005673,0.004802,0.004802,0.001094,0.000752


In [174]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Modèle d'arbre de décision
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

# Prédictions
y_pred = model.predict(X_test)

In [175]:
# Évaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

from sklearn.metrics import f1_score
f1_scores = f1_score(y_test, y_pred, average=None)  # `average=None` pour obtenir F1 pour chaque cible

# Calcul du F1-score moyen
f1_macro = f1_score(y_test, y_pred, average='macro')  # Moyenne des F1-scores

# Affichage des résultats
print(f"F1-score pour chaque cible: {f1_scores}")
print(f"F1-score moyen (macro): {f1_macro}")


Accuracy: 0.0
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00         4
           2       0.50      0.17      0.25         6
           3       0.50      0.14      0.22         7
           4       0.00      0.00      0.00         1
           5       0.00      0.00      0.00         0
           6       0.00      0.00      0.00         3
           7       0.00      0.00      0.00         1
           8       0.00      0.00      0.00         7
           9       0.00      0.00      0.00         4
          10       0.00      0.00      0.00         3
          11       0.00      0.00      0.00         2
          12       0.00      0.00      0.00         0
          13       0.00      0.00      0.00         0
          14       0.00      0.00      0.00         0
          15       0.00      0.00      0.00         0
          16       0.00      0.00      0.00         0
          17 

/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-pac